# Mr. Chicken - Lip-Sync com Wav2Lip (Google Colab)

Este notebook foi gerado pelo **Mr. Chicken** para permitir a sincronização labial gratuita usando o processamento em GPU do Google Colab.

### **Instruções de Uso:**
1. Vá no menu superior e clique em **Ambiente de execução** -> **Alterar tipo de ambiente de execução** e certifique-se de que a **GPU** (T4 GPU) está selecionada.
2. Execute as células abaixo em ordem clicando no botão **Play [ ]** à esquerda de cada célula.
3. Faça o upload do **Avatar** e do **Áudio de Voz** quando solicitado.
4. Aguarde o processamento terminar e faça o download do arquivo `result.mp4` gerado.
5. Envie o arquivo final de volta no painel do Mr. Chicken para concluir a renderização!

### **Passo 1: Instalar e configurar o Wav2Lip**

In [ ]:
import os
import shutil

# 1. Resetar diretório de trabalho no Colab para evitar clonagem aninhada recursiva
if os.path.exists('/content'):
    %cd /content

# 2. Clonar o repositório apenas se não existir
if not os.path.exists('Wav2Lip'):
    !git clone https://github.com/Rudrabha/Wav2Lip.git

%cd Wav2Lip

# 3. Criar diretório para checkpoints
checkpoint_path = "checkpoints/wav2lip_gan.pth"
os.makedirs("checkpoints", exist_ok=True)

# Lista de comandos de download com mirrors de fallback (Hugging Face via curl + Google Drive via gdown)
download_commands = [
    # Mirror 1: Hugging Face snehilsanyal usando curl (bypassa 401 com User-Agent)
    'curl -L -A "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36" "https://huggingface.co/snehilsanyal/Wav2Lip/resolve/main/checkpoints/wav2lip_gan.pth" -o checkpoints/wav2lip_gan.pth',
    
    # Mirror 2: Hugging Face briaai usando curl
    'curl -L -A "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36" "https://huggingface.co/briaai/Wav2Lip/resolve/main/wav2lip_gan.pth" -o checkpoints/wav2lip_gan.pth',
    
    # Mirror 3: Google Drive 1 (gdown)
    'gdown --id 1dwHujX7RVNCvdR1RR93z0FS2T2yzqup9 --output checkpoints/wav2lip_gan.pth',
    
    # Mirror 4: Google Drive 2 (gdown)
    'gdown --id 10Iu05Modfti3pDbxCFPnofmfVlbkvrCm --output checkpoints/wav2lip_gan.pth'
]

download_success = False
for cmd in download_commands:
    print(f"Executando download via mirror: {cmd[:60]}...")
    try:
        if os.path.exists(checkpoint_path):
            os.remove(checkpoint_path)
            
        # Executa o comando de terminal de download
        os.system(cmd)
        
        # Verifica se o arquivo baixado existe e tem tamanho esperado (> 100MB)
        if os.path.exists(checkpoint_path):
            size_mb = os.path.getsize(checkpoint_path) / (1024 * 1024)
            if size_mb > 100:
                print(f"✅ Download do checkpoint concluído com sucesso ({size_mb:.1f} MB)!")
                download_success = True
                break
            else:
                print(f"⚠️ Arquivo muito pequeno ({size_mb:.4f} MB). Provavelmente LFS pointer ou redirecionamento falho. Tentando próximo mirror...")
        else:
            print("⚠️ Arquivo não foi criado. Tentando próximo mirror...")
    except Exception as e:
        print(f"❌ Erro ao baixar deste mirror: {e}")

if not download_success:
    raise RuntimeError("Não foi possível baixar o checkpoint de nenhum dos mirrors públicos. Verifique se o Hugging Face ou Google Drive estão acessíveis.")

# 4. Corrigir incompatibilidades de bibliotecas antigas com o NumPy moderno no arquivo principal
with open("inference.py", "r") as f:
    inf_content = f.read()

# Injeta o patch do NumPy no topo do inference.py
numpy_patch = "import numpy as np\nnp.complex = complex\nnp.float = float\nnp.int = int\n"
if "np.complex = complex" not in inf_content:
    inf_content = numpy_patch + inf_content

# Corrige a definição do argumento --static que foi declarado incorretamente como type=bool
inf_content = inf_content.replace("type=bool, default=False", "action='store_true'")
inf_content = inf_content.replace("type=bool,default=False", "action='store_true'")
inf_content = inf_content.replace("type=bool", "action='store_true'")

with open("inference.py", "w") as f:
    f.write(inf_content)

# Corrige np.int por int no arquivo audio.py
with open("audio.py", "r") as f:
    audio_content = f.read()

audio_content = audio_content.replace("np.int", "int")

with open("audio.py", "w") as f:
    f.write(audio_content)

print("✅ Repositório clonado, corrigido e adaptado com sucesso!")

### **Passo 2: Baixar requerimentos adicionais**

In [ ]:
# Instalar requisitos do Wav2Lip
!pip install -q -r requirements.txt
# Instalar librosa antigo que é compatível com Wav2Lip ou ajustar
!pip install -q librosa==0.8.0
print("✅ Dependências instaladas com sucesso!")

### **Passo 3: Fazer Upload do Avatar e do Áudio do Mr. Chicken**
Execute a célula abaixo. Ela abrirá caixas de seleção de arquivos. Selecione:
1. O arquivo do avatar (imagem `.jpg`/`.png` ou vídeo `.mp4`).
2. O arquivo de áudio de voz gerado (`.wav` ou `.mp3`).

In [ ]:
from google.colab import files
import os
import shutil

# Limpar uploads anteriores se houver
for f in ['input_avatar', 'input_audio']:
    for ext in ['.png', '.jpg', '.jpeg', '.webp', '.mp4', '.avi', '.mov', '.wav', '.mp3']:
        if os.path.exists(f + ext):
            os.remove(f + ext)

print("1. FAÇA UPLOAD DO AVATAR (Vídeo ou Imagem):")
up_avatar = files.upload()
avatar_name = list(up_avatar.keys())[0]
avatar_ext = os.path.splitext(avatar_name)[1].lower()
shutil.move(avatar_name, "input_avatar" + avatar_ext)

print("\n2. FAÇA UPLOAD DO ÁUDIO DE VOZ:")
up_audio = files.upload()
audio_name = list(up_audio.keys())[0]
audio_ext = os.path.splitext(audio_name)[1].lower()
shutil.move(audio_name, "input_audio" + audio_ext)

# Converter áudio para .wav para garantir 100% de compatibilidade
!ffmpeg -y -i "input_audio{audio_ext}" -ac 1 -ar 16000 input_audio_processed.wav

print("\n✅ Arquivos carregados e preparados!")

### **Passo 4: Executar a Sincronização Labial (Lip-Sync)**
Execute esta célula para rodar o modelo Wav2Lip.

In [ ]:
import glob

# Identificar arquivo de avatar
avatar_files = glob.glob("input_avatar.*")[0]
avatar_ext = os.path.splitext(avatar_files)[1].lower()
is_image = avatar_ext in ['.png', '.jpg', '.jpeg', '.webp']

cmd = f"python inference.py --checkpoint_path checkpoints/wav2lip_gan.pth --face {avatar_files} --audio input_audio_processed.wav --outfile result.mp4"
if is_image:
    cmd += " --static"

print(f"Executando comando: {cmd}")
!{cmd}

if os.path.exists("result.mp4"):
    print("\n✅ Lip-sync concluído com sucesso! result.mp4 gerado.")
else:
    print("\n❌ Falha ao gerar o vídeo. Verifique se o rosto no avatar está bem visível e frontal.")

### **Passo 5: Fazer o Download do Vídeo Final**

In [ ]:
if os.path.exists("result.mp4"):
    print("Iniciando download do vídeo...")
    files.download("result.mp4")
else:
    print("Arquivo result.mp4 não encontrado.")

### **Passo 6: Modo Fila Automática (Geração em Massa - Runner)**

Use este modo se você deseja criar múltiplos vídeos em massa sem precisar ficar carregando e baixando arquivos manualmente a todo momento.

1. Preencha as credenciais do seu Supabase abaixo (copie-as do seu arquivo `.env.local`).
2. Execute a célula abaixo. Ela ficará rodando em loop contínuo, consultando novos vídeos criados no painel, fazendo a dublagem automática e enviando os resultados de volta para o Next.js finalizar a renderização.

In [ ]:
# Instalar SDK do Supabase
!pip install -q supabase

import time
import os
import glob
from supabase import create_client, Client

# @title Insira as credenciais do Supabase para processamento automático
SUPABASE_URL = "" # @param {type:"string"}
SUPABASE_KEY = "" # @param {type:"string"}

if not SUPABASE_URL or not SUPABASE_KEY:
    raise ValueError("Por favor, insira a URL e a Chave Anon do seu Supabase para ativar a automação.")

supabase: Client = create_client(SUPABASE_URL, SUPABASE_KEY)

print("🚀 Runner de Fila do Mr. Chicken iniciado com sucesso!")
print("Aguardando jobs pendentes de lip-sync... (Pressione 'Stop' para interromper)")

while True:
    try:
        # 1. Buscar um job que esteja com status 'lip_syncing' e sem lip_sync_video_path
        res = supabase.table("reaction_jobs").select("*, avatars(*)").eq("status", "lip_syncing").is_("lip_sync_video_path", "null").limit(1).execute()
        jobs = res.data
        
        if jobs:
            job = jobs[0]
            job_id = job["id"]
            avatar = job["avatars"]
            
            print(f"\n[Fila] Processando Job: {job_id} (Assunto: {job['topic']})")
            
            # Limpar arquivos temporários anteriores
            for f in glob.glob("temp_avatar.*") + glob.glob("temp_audio.*") + ["result.mp4", "input_audio_processed.wav"]:
                if os.path.exists(f):
                    os.remove(f)
            
            # 2. Baixar o arquivo do avatar base
            avatar_path = avatar["image_path"]
            avatar_ext = os.path.splitext(avatar_path)[1].lower() or ".mp4"
            local_avatar = f"temp_avatar{avatar_ext}"
            print(f"-> Baixando avatar base de: {avatar_path}...")
            
            with open(local_avatar, "wb") as f:
                res_storage = supabase.storage.from_("avatars").download(avatar_path)
                f.write(res_storage)
                
            # 3. Baixar o áudio de voz gerado
            audio_path = job["audio_path"]
            audio_ext = os.path.splitext(audio_path)[1].lower() or ".mp3"
            local_audio = f"temp_audio{audio_ext}"
            print(f"-> Baixando áudio de locução de: {audio_path}...")
            
            with open(local_audio, "wb") as f:
                res_storage = supabase.storage.from_("job-assets").download(audio_path)
                f.write(res_storage)
                
            # Converter o áudio para WAV 16kHz mono (formato esperado pelo Wav2Lip)
            os.system(f'ffmpeg -y -i "{local_audio}" -ac 1 -ar 16000 input_audio_processed.wav > /dev/null 2>&1')
            
            # 4. Executar Wav2Lip
            is_image = avatar_ext in ['.png', '.jpg', '.jpeg', '.webp']
            cmd = f"python inference.py --checkpoint_path checkpoints/wav2lip_gan.pth --face {local_avatar} --audio input_audio_processed.wav --outfile result.mp4"
            if is_image:
                cmd += " --static"
                
            print(f"-> Rodando inferência da rede neural Wav2Lip (GPU)... (Aguarde)")
            os.system(cmd)
            
            if os.path.exists("result.mp4"):
                print("-> Lip-sync gerado! Enviando vídeo para o Supabase Storage...")
                
                # 5. Fazer upload do resultado de volta para o bucket 'job-assets'
                supabase_lipsync_path = f"{job['user_id']}/{job_id}-lipsync.mp4"
                
                with open("result.mp4", "rb") as f:
                    supabase.storage.from_("job-assets").upload(
                        path=supabase_lipsync_path,
                        file=f,
                        file_options={"content-type": "video/mp4", "upsert": "true"}
                    )
                
                # 6. Atualizar a coluna lip_sync_video_path no banco de dados
                supabase.table("reaction_jobs").update({
                    "lip_sync_video_path": supabase_lipsync_path
                }).eq("id", job_id).execute()
                
                print(f"✅ Vídeo sincronizado! O Next.js iniciará o render vertical automático.")
            else:
                print("❌ Erro: O arquivo result.mp4 não foi gerado.")
                supabase.table("reaction_jobs").update({
                    "status": "failed",
                    "error_message": "Wav2Lip falhou ao gerar o vídeo na nuvem do Colab."
                }).eq("id", job_id).execute()
                
        else:
            # Feedback visual de atividade
            print(".", end="", flush=True)
            
    except Exception as e:
        print(f"\n❌ Ocorreu um erro no processamento: {e}")
        
    time.sleep(10)

### **Passo 7: Modo API Automática (Gradio + Next.js)**

Use este modo para que o Next.js se comunique diretamente com o Google Colab via API sem que você precise fazer upload/download manual de arquivos ou configurar o Supabase.

1. Execute a célula abaixo.
2. Copie o link público temporário gerado (ex: `https://xxxx.gradio.live`).
3. Cole este link no seu arquivo `.env.local` na variável `LIPSYNC_API_URL` do seu projeto Mr. Chicken.
4. Inicie um job! A sincronização será totalmente automática.

In [ ]:
!pip install -q gradio

import gradio as gr
import subprocess
import shutil
import os

def api_lipsync(face_file, audio_file):
    output_path = "api_result.mp4"
    if os.path.exists(output_path):
        os.remove(output_path)
        
    # 1. Obter extensões e caminhos
    face_ext = os.path.splitext(face_file.name)[1].lower()
    audio_ext = os.path.splitext(audio_file.name)[1].lower()
    
    # 2. Copiar/preparar arquivos locais
    local_face = f"api_face{face_ext}"
    local_audio = f"api_audio{audio_ext}"
    
    if os.path.exists(local_face): os.remove(local_face)
    if os.path.exists(local_audio): os.remove(local_audio)
    if os.path.exists("input_audio_processed.wav"): os.remove("input_audio_processed.wav")
    
    shutil.copy(face_file.name, local_face)
    shutil.copy(audio_file.name, local_audio)
    
    # Converter o áudio para WAV 16kHz mono (formato esperado pelo Wav2Lip)
    subprocess.run(f'ffmpeg -y -i "{local_audio}" -ac 1 -ar 16000 input_audio_processed.wav', shell=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    
    # 3. Executar Wav2Lip
    is_image = face_ext in ['.png', '.jpg', '.jpeg', '.webp']
    cmd = f"python inference.py --checkpoint_path checkpoints/wav2lip_gan.pth --face {local_face} --audio input_audio_processed.wav --outfile {output_path}"
    if is_image:
        cmd += " --static"
        
    print(f"-> Processando Lip-Sync: {cmd}")
    subprocess.run(cmd, shell=True)
    
    if os.path.exists(output_path):
        print("✅ Lip-sync processado com sucesso!")
        return output_path
    else:
        raise RuntimeError("Falha ao gerar o vídeo de lip-sync no Colab.")

print("🚀 Iniciando servidor de API do Mr. Chicken...")
demo = gr.Interface(
    fn=api_lipsync,
    inputs=[
        gr.File(type="filepath", label="Avatar (Imagem ou Vídeo)"),
        gr.File(type="filepath", label="Voz (Áudio)")
    ],
    outputs=gr.File(label="Vídeo Final Sincronizado"),
    title="Mr. Chicken Lip-Sync API Server"
)
demo.launch(share=True)